In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F


def occurrences_enriched():
    incidents = spark.read.table(
        "workspace.silver.occurrence_clean"
    ).alias("i")

    companies = (
        spark.read.table("workspace.bronze.company_raw")
        .select(
            F.col("id").cast("string").alias("dim_company_id"),
            F.trim(F.col("name")).alias("company_name"),
        )
        .alias("co")
    )

    areas = (
        spark.read.table("workspace.bronze.area_raw")
        .select(
            F.col("id").cast("string").alias("dim_area_id"),
            F.trim(F.col("sector_name")).alias("sector_name"),
        )
        .alias("ar")
    )

    waste_types = (
        spark.read.table("workspace.bronze.waste_type_raw")
        .select(
            F.col("id").cast("string").alias("dim_waste_type_id"),
            F.trim(F.col("category")).alias("waste_category"),
        )
        .alias("wt")
    )

    return (
        incidents
        .join(
            companies,
            F.col("i.company_id") == F.col("co.dim_company_id"),
            "left",
        )
        .join(
            areas,
            F.col("i.area_id") == F.col("ar.dim_area_id"),
            "left",
        )
        .join(
            waste_types,
            F.col("i.waste_type_id") == F.col("wt.dim_waste_type_id"),
            "left",
        )
        .select(
            "i.*",
            "co.company_name",
            "ar.sector_name",
            "wt.waste_category",
        )
    )


def collections_enriched():
    # Na Silver, a data da solicitação chama-se requested_date_key.
    # O Gold padroniza o nome para date_key.
    collections = (
        spark.read.table("workspace.silver.collection_clean")
        .withColumnRenamed("requested_date_key", "date_key")
        .alias("c")
    )

    companies = (
        spark.read.table("workspace.bronze.company_raw")
        .select(
            F.col("id").cast("string").alias("dim_company_id"),
            F.trim(F.col("name")).alias("company_name"),
        )
        .alias("co")
    )

    areas = (
        spark.read.table("workspace.bronze.area_raw")
        .select(
            F.col("id").cast("string").alias("dim_area_id"),
            F.trim(F.col("sector_name")).alias("sector_name"),
        )
        .alias("ar")
    )

    waste_types = (
        spark.read.table("workspace.bronze.waste_type_raw")
        .select(
            F.col("id").cast("string").alias("dim_waste_type_id"),
            F.trim(F.col("category")).alias("waste_category"),
        )
        .alias("wt")
    )

    cooperatives = (
        spark.read.table("workspace.bronze.cooperative_raw")
        .select(
            F.col("id").cast("string").alias("dim_cooperative_id"),
            F.trim(F.col("name")).alias("cooperative_name"),
            F.col("latitude").cast("double").alias("latitude"),
            F.col("longitude").cast("double").alias("longitude"),
        )
        .alias("cp")
    )

    return (
        collections
        .join(
            companies,
            F.col("c.company_id") == F.col("co.dim_company_id"),
            "left",
        )
        .join(
            areas,
            F.col("c.area_id") == F.col("ar.dim_area_id"),
            "left",
        )
        .join(
            waste_types,
            F.col("c.waste_type_id") == F.col("wt.dim_waste_type_id"),
            "left",
        )
        .join(
            cooperatives,
            F.col("c.cooperative_id") == F.col("cp.dim_cooperative_id"),
            "left",
        )
        .select(
            "c.*",
            "co.company_name",
            "ar.sector_name",
            "wt.waste_category",
            "cp.cooperative_name",
            "cp.latitude",
            "cp.longitude",
        )
    )


@dp.materialized_view(
    name="workspace.gold.occurrences_by_sector",
    comment="Ocorrências por data, empresa e setor."
)
def occurrences_by_sector():
    data = occurrences_enriched()

    return (
        data
        .groupBy(
            "date_key",
            "company_id",
            "company_name",
            "area_id",
            "sector_name",
            "priority",
            "status",
            "waste_category",
        )
        .agg(
            F.countDistinct("incident_id").alias("occurrence_count"),
            F.coalesce(
                F.sum("estimated_quantity_kg"),
                F.lit(0),
            ).alias("estimated_quantity_kg"),
        )
    )


@dp.materialized_view(
    name="workspace.gold.occurrences_over_time",
    comment="Série temporal diária das ocorrências registradas."
)
def occurrences_over_time():
    data = occurrences_enriched()

    return (
        data
        .groupBy(
            "date_key",
            "company_id",
            "company_name",
            "priority",
            "status",
            "waste_category",
        )
        .agg(
            F.countDistinct("incident_id").alias("occurrence_count"),
            F.coalesce(
                F.sum("estimated_quantity_kg"),
                F.lit(0),
            ).alias("estimated_quantity_kg"),
        )
    )


@dp.materialized_view(
    name="workspace.gold.occurrences_by_hour",
    comment="Distribuição de ocorrências por hora local."
)
def occurrences_by_hour():
    data = occurrences_enriched()

    return (
        data
        .groupBy(
            "date_key",
            "company_id",
            "company_name",
            "hour_of_day",
            "priority",
            "status",
            "waste_category",
        )
        .agg(
            F.countDistinct("incident_id").alias("occurrence_count")
        )
    )


@dp.materialized_view(
    name="workspace.gold.collection_resolution",
    comment=(
        "Coletas com evento histórico de conclusão e tempo válido "
        "para análise de resolução."
    )
)
def collection_resolution():
    data = collections_enriched()

    return (
        data
        .filter(
            F.col("has_completion_event")
            & F.col("resolution_time_hours").isNotNull()
        )
        .select(
            "collection_id",
            "date_key",
            "company_id",
            "company_name",
            "area_id",
            "sector_name",
            "waste_category",
            "cooperative_id",
            "cooperative_name",
            "requested_at_local",
            "completed_at_local",
            "resolution_time_hours",
        )
    )


@dp.materialized_view(
    name="workspace.gold.occurrences_by_cooperative",
    comment=(
        "Contagem por cooperativa; as coordenadas representam "
        "a cooperativa associada."
    )
)
def occurrences_by_cooperative():
    data = collections_enriched()

    return (
        data
        .filter(
            F.col("cooperative_id").isNotNull()
            & F.col("latitude").isNotNull()
            & F.col("longitude").isNotNull()
        )
        .groupBy(
            "date_key",
            "company_id",
            "company_name",
            "cooperative_id",
            "cooperative_name",
            "latitude",
            "longitude",
        )
        .agg(
            F.countDistinct("incident_id").alias("occurrence_count"),
            F.countDistinct("collection_id").alias("collection_count"),
        )
    )


@dp.materialized_view(
    name="workspace.gold.operational_kpis",
    comment=(
        "Componentes diários: conclusão histórica, status atual "
        "e tempo de resolução."
    )
)
def operational_kpis():
    occurrences = (
        spark.read.table("workspace.silver.occurrence_clean")
        .groupBy("date_key", "company_id")
        .agg(
            F.countDistinct("incident_id").alias("total_occurrences"),
            F.coalesce(
                F.sum("estimated_quantity_kg"),
                F.lit(0),
            ).alias("estimated_quantity_kg"),
        )
    )

    collections_source = (
        spark.read.table("workspace.silver.collection_clean")
        .withColumnRenamed("requested_date_key", "date_key")
    )

    collections = (
        collections_source
        .groupBy("date_key", "company_id")
        .agg(
            F.countDistinct("collection_id").alias("total_collections"),

            # Concluídas alguma vez, segundo o histórico de status.
            F.countDistinct(
                F.when(
                    F.col("has_completion_event"),
                    F.col("collection_id"),
                )
            ).alias("completed_collections"),

            # Concluídas no status atual; indicador separado.
            F.countDistinct(
                F.when(
                    F.col("is_currently_completed"),
                    F.col("collection_id"),
                )
            ).alias("currently_completed_collections"),

            # Soma e denominador só incluem durações válidas
            # associadas a um evento histórico de conclusão.
            F.coalesce(
                F.sum(
                    F.when(
                        F.col("has_completion_event")
                        & F.col("resolution_time_hours").isNotNull(),
                        F.col("resolution_time_hours"),
                    ).otherwise(F.lit(0.0))
                ),
                F.lit(0.0),
            ).alias("resolution_hours_sum"),

            F.countDistinct(
                F.when(
                    F.col("has_completion_event")
                    & F.col("resolution_time_hours").isNotNull(),
                    F.col("collection_id"),
                )
            ).alias("resolved_collections_count"),
        )
    )

    daily = occurrences.join(
        collections,
        ["date_key", "company_id"],
        "full_outer",
    )

    return daily.select(
        "date_key",
        "company_id",
        F.coalesce("total_occurrences", F.lit(0))
            .alias("total_occurrences"),
        F.coalesce("estimated_quantity_kg", F.lit(0))
            .alias("estimated_quantity_kg"),
        F.coalesce("total_collections", F.lit(0))
            .alias("total_collections"),
        F.coalesce("completed_collections", F.lit(0))
            .alias("completed_collections"),
        F.coalesce("currently_completed_collections", F.lit(0))
            .alias("currently_completed_collections"),
        F.coalesce("resolution_hours_sum", F.lit(0.0))
            .alias("resolution_hours_sum"),
        F.coalesce("resolved_collections_count", F.lit(0))
            .alias("resolved_collections_count"),
    )